# Lecture 8: Locality, EPR, and Bell's Theorem
### Computational Companion — No-Signaling, CHSH Inequality, and the Tsirelson Bound

This notebook verifies every claim in Lecture 8:

1. **No-signaling theorem** — Bob's operations leave Alice's ρ unchanged
2. **EPR entropy check** — composite S=0 but subsystem S=1 bit
3. **Singlet correlation formula** — $\langle (\vec{\sigma}\cdot\hat{a})\otimes(\vec{\sigma}\cdot\hat{b})\rangle = -\hat{a}\cdot\hat{b}$
4. **CHSH classical bound** — exhaustive check over all ±1 assignments
5. **CHSH quantum violation** — $|S| = 2\sqrt{2} \approx 2.828 > 2$
6. **Tsirelson bound** — angle sweep confirms $2\sqrt{2}$ is the maximum
7. **Decoherence and measurement** — CNOT + partial trace kills coherences

**Convention:** ℏ = 1 throughout.

**Reference:** Lecture 8 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from itertools import product as iter_product

# ── Pauli matrices ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

e0 = np.array([1, 0], dtype=complex)
e1 = np.array([0, 1], dtype=complex)

def kron(a, b):
    return np.kron(a, b)

def partial_trace_B(rho_AB, dA=2, dB=2):
    rho_A = np.zeros((dA, dA), dtype=complex)
    for i in range(dA):
        for j in range(dA):
            for k in range(dB):
                rho_A[i, j] += rho_AB[i*dB + k, j*dB + k]
    return rho_A

def expect(M, v):
    return np.real(v.conj() @ M @ v)

def von_neumann_entropy(rho):
    evals = np.linalg.eigvalsh(rho)
    evals = evals[evals > 1e-15]
    return -np.sum(evals * np.log2(evals))

# Singlet state
v_singlet = (kron(e0, e1) - kron(e1, e0)) / np.sqrt(2)

print("Setup complete.")

## 1. EPR Entropy Check

The singlet is a pure composite state ($S_{\text{composite}} = 0$) but each subsystem has $S = \log_2 2 = 1$ bit (maximally mixed). We verify this numerically.

In [ ]:
# ── 1. EPR Entropy Check ──

print("=" * 65)
print("EPR: composite S=0, subsystem S=1 bit")
print("=" * 65)

rho_AB = np.outer(v_singlet, v_singlet.conj())
S_composite = von_neumann_entropy(rho_AB)

rho_A = partial_trace_B(rho_AB)
S_A = von_neumann_entropy(rho_A)

print(f"\nComposite state:")
print(f"  Eigenvalues of ρ_AB: {np.round(np.linalg.eigvalsh(rho_AB), 6)}")
print(f"  S(composite) = {S_composite:.6f} bits (pure state → zero ignorance)")

print(f"\nAlice's subsystem:")
print(f"  ρ^(A) = {np.round(rho_A, 4)}")
print(f"  Eigenvalues: {np.round(np.linalg.eigvalsh(rho_A), 4)}")
print(f"  S(A) = {S_A:.6f} bits (maximally mixed → maximum ignorance)")

print(f"\n→ The whole is in a state of ZERO uncertainty (S=0).")
print(f"  Each part is in a state of MAXIMUM uncertainty (S=1 bit).")
print(f"  This is the EPR observation in mathematical form.")

## 2. Singlet Correlation Formula

For the singlet: $\langle (\vec{\sigma}\cdot\hat{a}) \otimes (\vec{\sigma}\cdot\hat{b}) \rangle = -\hat{a} \cdot \hat{b}$.

We verify this for many pairs of directions.

In [ ]:
# ── 2. Singlet Correlation Formula ──

print("=" * 65)
print("SINGLET CORRELATION: ⟨(σ·â)⊗(σ·b̂)⟩ = -â·b̂")
print("=" * 65)

def spin_obs(n_hat):
    """Spin observable σ·n̂ for unit vector n̂ = (nx, ny, nz)."""
    return n_hat[0]*X + n_hat[1]*Y + n_hat[2]*Z

# Test specific directions
directions = [
    ("ẑ", np.array([0, 0, 1.0])),
    ("x̂", np.array([1, 0, 0.0])),
    ("ŷ", np.array([0, 1, 0.0])),
    ("(ẑ+x̂)/√2", np.array([1, 0, 1.0]) / np.sqrt(2)),
    ("(ẑ−x̂)/√2", np.array([-1, 0, 1.0]) / np.sqrt(2)),
]

print(f"\n  {'â':<12} {'b̂':<12} {'⟨(σ·â)⊗(σ·b̂)⟩':>18} {'−â·b̂':>8} {'Match'}")
print("  " + "-" * 65)
for name_a, a in directions:
    for name_b, b in directions:
        obs = kron(spin_obs(a), spin_obs(b))
        val = expect(obs, v_singlet)
        expected = -np.dot(a, b)
        print(f"  {name_a:<12} {name_b:<12} {val:18.6f} {expected:8.4f}   {np.isclose(val, expected)}")

# Random directions sweep
rng = np.random.default_rng(42)
n_tests = 1000
max_err = 0
for _ in range(n_tests):
    a = rng.standard_normal(3); a /= np.linalg.norm(a)
    b = rng.standard_normal(3); b /= np.linalg.norm(b)
    val = expect(kron(spin_obs(a), spin_obs(b)), v_singlet)
    max_err = max(max_err, abs(val - (-np.dot(a, b))))
print(f"\n  {n_tests} random direction pairs: max error = {max_err:.2e} ✓")

## 3. CHSH Classical Bound — Exhaustive Verification

For any local hidden variable assignment $A_0, A_1, B_0, B_1 \in \{\pm 1\}$, the CHSH expression $A_0(B_0+B_1) + A_1(B_0-B_1)$ is bounded by 2. We verify by checking all $2^4 = 16$ assignments.

In [ ]:
# ── 3. CHSH Classical Bound ──

print("=" * 65)
print("CHSH CLASSICAL BOUND: exhaustive check over all ±1 assignments")
print("=" * 65)

print(f"\n  {'A0':>3} {'A1':>3} {'B0':>3} {'B1':>3} | {'A0(B0+B1)':>10} {'A1(B0-B1)':>10} {'S':>5}")
print("  " + "-" * 50)

max_S = 0
for a0, a1, b0, b1 in iter_product([1, -1], repeat=4):
    term1 = a0 * (b0 + b1)
    term2 = a1 * (b0 - b1)
    S = term1 + term2
    max_S = max(max_S, abs(S))
    print(f"  {a0:3d} {a1:3d} {b0:3d} {b1:3d} | {term1:10d} {term2:10d} {S:5d}")

print(f"\n  Maximum |S| over all assignments: {max_S}")
print(f"  CHSH inequality: |S| ≤ {max_S} ✓")
print(f"\n  For ANY probability distribution over λ:")
print(f"  |⟨S⟩| ≤ max|S| = 2 (convex combination of ±2 values)")

## 4. CHSH Quantum Violation — $|S| = 2\sqrt{2}$

Using the optimal measurement directions at 45° spacing:
- Alice: $A_0 = Z$, $A_1 = X$
- Bob: $B_0 = \frac{Z+X}{\sqrt{2}}$, $B_1 = \frac{Z-X}{\sqrt{2}}$

We compute the CHSH quantity in the singlet and verify $|S| = 2\sqrt{2}$.

In [ ]:
# ── 4. CHSH Quantum Violation ──

print("=" * 65)
print("CHSH QUANTUM VIOLATION: |S| = 2√2")
print("=" * 65)

# Measurement directions (matching lecture §4.4)
a0_hat = np.array([0, 0, 1.0])  # ẑ
a1_hat = np.array([1, 0, 0.0])  # x̂
b0_hat = np.array([1, 0, 1.0]) / np.sqrt(2)  # (ẑ+x̂)/√2
b1_hat = np.array([-1, 0, 1.0]) / np.sqrt(2)  # (ẑ−x̂)/√2

A0 = spin_obs(a0_hat)
A1 = spin_obs(a1_hat)
B0 = spin_obs(b0_hat)
B1 = spin_obs(b1_hat)

# Verify eigenvalues ±1
for name, op in [('A0', A0), ('A1', A1), ('B0', B0), ('B1', B1)]:
    evals = np.round(np.linalg.eigvalsh(op), 4)
    print(f"  {name} eigenvalues: {evals}")

# Compute each correlation
c00 = expect(kron(A0, B0), v_singlet)
c01 = expect(kron(A0, B1), v_singlet)
c10 = expect(kron(A1, B0), v_singlet)
c11 = expect(kron(A1, B1), v_singlet)

print(f"\n  Using singlet correlation formula ⟨(σ·â)⊗(σ·b̂)⟩ = −â·b̂:")
print(f"  ⟨A₀B₀⟩ = −â₀·b̂₀ = {c00:.6f}  (expected: {-np.dot(a0_hat, b0_hat):.6f})")
print(f"  ⟨A₀B₁⟩ = −â₀·b̂₁ = {c01:.6f}  (expected: {-np.dot(a0_hat, b1_hat):.6f})")
print(f"  ⟨A₁B₀⟩ = −â₁·b̂₀ = {c10:.6f}  (expected: {-np.dot(a1_hat, b0_hat):.6f})")
print(f"  ⟨A₁B₁⟩ = −â₁·b̂₁ = {c11:.6f}  (expected: {-np.dot(a1_hat, b1_hat):.6f})")

S = c00 + c01 + c10 - c11
print(f"\n  S = {c00:.4f} + {c01:.4f} + {c10:.4f} − {c11:.4f} = {S:.6f}")
print(f"  |S| = {abs(S):.6f}")
print(f"  2√2 = {2*np.sqrt(2):.6f}")
print(f"  |S| = 2√2? {np.isclose(abs(S), 2*np.sqrt(2))}")
print(f"\n  Classical bound: |S| ≤ 2")
print(f"  Quantum value:   |S| = {abs(S):.4f} > 2")
print(f"  ╔═══════════════════════════════════════════════╗")
print(f"  ║  BELL INEQUALITY VIOLATED by factor √2       ║")
print(f"  ║  No local hidden variable theory works.       ║")
print(f"  ║  2022 Nobel Prize in Physics.                 ║")
print(f"  ╚═══════════════════════════════════════════════╝")

## 5. Tsirelson Bound — Angle Sweep

We sweep over all measurement angles in the $xz$-plane to confirm that $2\sqrt{2}$ is the maximum quantum CHSH value. We also compare against the classical bound of 2 and the algebraic maximum of 4.

In [ ]:
# ── 5. Tsirelson Bound ──

print("=" * 65)
print("TSIRELSON BOUND: sweeping measurement angles")
print("=" * 65)

# Fix Alice's settings, sweep Bob's angle θ in xz-plane
# Bob: b0 at angle θ from z, b1 at angle θ+π/2 from z
thetas = np.linspace(0, 2*np.pi, 500)
S_vals = np.zeros(len(thetas))

for i, th in enumerate(thetas):
    b0 = np.array([np.sin(th), 0, np.cos(th)])
    b1 = np.array([np.sin(th + np.pi/2), 0, np.cos(th + np.pi/2)])
    S_vals[i] = (expect(kron(A0, spin_obs(b0)), v_singlet) +
                 expect(kron(A0, spin_obs(b1)), v_singlet) +
                 expect(kron(A1, spin_obs(b0)), v_singlet) -
                 expect(kron(A1, spin_obs(b1)), v_singlet))

plt.figure(figsize=(12, 6))
plt.plot(np.degrees(thetas), S_vals, 'steelblue', linewidth=2, label='S(θ)')
plt.axhline(2, color='red', ls='--', lw=2, label='Classical bound = 2')
plt.axhline(-2, color='red', ls='--', lw=2)
plt.axhline(2*np.sqrt(2), color='green', ls=':', lw=2, label=f'Tsirelson bound = 2√2 ≈ {2*np.sqrt(2):.3f}')
plt.axhline(-2*np.sqrt(2), color='green', ls=':', lw=2)
plt.axhline(4, color='gray', ls='-.', lw=1, alpha=0.5, label='Algebraic max = 4 (PR box)')
plt.axhline(-4, color='gray', ls='-.', lw=1, alpha=0.5)
plt.fill_between(np.degrees(thetas), -2, 2, alpha=0.1, color='red', label='Classical region')
plt.xlabel("Bob's angle θ (degrees)"); plt.ylabel('CHSH value S')
plt.title("CHSH value vs Bob's measurement angle\nAlice: A₀=Z, A₁=X; Bob: b₀ at θ, b₁ at θ+90°")
plt.legend(loc='upper right', fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Maximum |S| found: {np.max(np.abs(S_vals)):.6f}")
print(f"Tsirelson bound:   {2*np.sqrt(2):.6f}")
print(f"Achieved at θ ≈ {np.degrees(thetas[np.argmax(np.abs(S_vals))]):.1f}°")

# Full 2D sweep: vary both Alice and Bob angles
n_ang = 100
alice_angles = np.linspace(0, np.pi, n_ang)
bob_angles = np.linspace(0, np.pi, n_ang)
S_2d = np.zeros((n_ang, n_ang))

for i, alpha in enumerate(alice_angles):
    a0_2d = np.array([0, 0, 1.0])  # fixed Z
    a1_2d = np.array([np.sin(alpha), 0, np.cos(alpha)])
    for j, beta in enumerate(bob_angles):
        b0_2d = np.array([np.sin(beta), 0, np.cos(beta)])
        b1_2d = np.array([np.sin(beta + np.pi/2), 0, np.cos(beta + np.pi/2)])
        S_2d[i, j] = (expect(kron(spin_obs(a0_2d), spin_obs(b0_2d)), v_singlet) +
                       expect(kron(spin_obs(a0_2d), spin_obs(b1_2d)), v_singlet) +
                       expect(kron(spin_obs(a1_2d), spin_obs(b0_2d)), v_singlet) -
                       expect(kron(spin_obs(a1_2d), spin_obs(b1_2d)), v_singlet))

plt.figure(figsize=(8, 6))
plt.imshow(np.abs(S_2d), extent=[0, 180, 180, 0], aspect='auto', cmap='hot',
           vmin=0, vmax=2*np.sqrt(2))
plt.colorbar(label='|S|')
plt.contour(bob_angles * 180/np.pi, alice_angles * 180/np.pi, np.abs(S_2d),
            levels=[2], colors='cyan', linewidths=2)
plt.xlabel("Bob's angle β (degrees)"); plt.ylabel("Alice's angle α (degrees)")
plt.title('|S| over Alice/Bob angle space\nCyan contour = classical bound |S|=2')
plt.tight_layout(); plt.show()

print(f"Global max |S| over 2D grid: {np.max(np.abs(S_2d)):.6f}")

## 6. No-Signaling Verification

We re-verify the no-signaling theorem from Lecture 7/8: Bob's unitary operations and measurements cannot change Alice's density matrix. We test on the singlet with several operations.

In [ ]:
# ── 6. No-Signaling Verification ──

print("=" * 65)
print("NO-SIGNALING: Bob's operations don't change Alice's ρ")
print("=" * 65)

rho_AB = np.outer(v_singlet, v_singlet.conj())
rho_A_orig = partial_trace_B(rho_AB)

# Bob unitaries
bob_ops = [
    ("I", I2), ("X", X), ("Y", Y), ("Z", Z),
    ("Hadamard", np.array([[1,1],[1,-1]], dtype=complex)/np.sqrt(2)),
    ("exp(iθσ)", expm(1j*(0.7*X + 0.3*Y + 0.5*Z))),
]

print("\nBob applies unitary (I⊗U_B):")
for name, UB in bob_ops:
    IUB = kron(I2, UB)
    rho_new = partial_trace_B(IUB @ rho_AB @ IUB.conj().T)
    diff = np.max(np.abs(rho_new - rho_A_orig))
    print(f"  {name:<15} max|Δρ| = {diff:.2e}  {'✓' if diff < 1e-12 else '✗'}")

# What if Bob's operation were NON-unitary?
print("\nWhat if Bob's evolution were NON-unitary?")
M_nonunitary = np.array([[1, 0], [0, 0.5]], dtype=complex)  # not unitary
print(f"  M = {M_nonunitary.tolist()}, M†M = {np.round((M_nonunitary.conj().T @ M_nonunitary), 4).tolist()}")
IM = kron(I2, M_nonunitary)
v_new = IM @ v_singlet
v_new /= np.linalg.norm(v_new)  # renormalize
rho_new = partial_trace_B(np.outer(v_new, v_new.conj()))
diff = np.max(np.abs(rho_new - rho_A_orig))
print(f"  Non-unitary Bob: max|Δρ| = {diff:.4f}  → Alice's ρ CHANGED! Signaling possible!")
print(f"  → This is why unitarity cannot be modified.")

## 7. Decoherence and the Arrow of Time

We simulate measurement-as-decoherence: a system in superposition entangles with an apparatus via CNOT, and the partial trace over the apparatus kills the off-diagonal coherences. We also show how the number of environment degrees of freedom affects decoherence.

In [ ]:
# ── 7. Decoherence and Arrow of Time ──

print("=" * 65)
print("DECOHERENCE: measurement kills coherences")
print("=" * 65)

# System in superposition
alpha = 1/np.sqrt(3)
beta = np.sqrt(2/3) * np.exp(1j * np.pi/4)
psi = alpha * e0 + beta * e1

# Before measurement: pure state
rho_pure = np.outer(psi, psi.conj())
print(f"\nBefore measurement (pure state):")
print(f"  ρ = {np.round(rho_pure, 4)}")
print(f"  Off-diagonal |ρ₀₁| = {abs(rho_pure[0,1]):.4f}")
print(f"  Purity = {np.real(np.trace(rho_pure @ rho_pure)):.4f}")

# After CNOT with 1 apparatus qubit
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
v_pre = kron(psi, e0)
v_post = CNOT @ v_pre
rho_sys_1 = partial_trace_B(np.outer(v_post, v_post.conj()))
print(f"\nAfter CNOT with 1 apparatus qubit:")
print(f"  ρ = {np.round(rho_sys_1, 4)}")
print(f"  Off-diagonal |ρ₀₁| = {abs(rho_sys_1[0,1]):.6f}")
print(f"  Purity = {np.real(np.trace(rho_sys_1 @ rho_sys_1)):.4f}")

# Multiple environment qubits: each one CNOTs with the system
print(f"\nMultiple environment qubits (successive CNOTs):")
n_env_list = [1, 2, 3, 4, 5]
coherences = []
purities = []

for n_env in n_env_list:
    # Build state: system ⊗ n_env qubits, all starting in |0⟩
    d_env = 2**n_env
    env_state = np.zeros(d_env, dtype=complex)
    env_state[0] = 1  # all |0⟩
    v_full = kron(psi, env_state)  # system ⊗ environment
    
    # Apply CNOT from system to each environment qubit
    d_total = 2 * d_env
    for q in range(n_env):
        # CNOT: system (qubit 0) controls environment qubit q+1
        # Build the gate on the full space
        gate = np.eye(d_total, dtype=complex)
        # For each basis state, if system=1, flip env qubit q
        for idx in range(d_total):
            sys_bit = idx >> n_env  # most significant bit = system
            if sys_bit == 1:
                env_bits = idx & (d_env - 1)
                flipped = env_bits ^ (1 << (n_env - 1 - q))
                new_idx = (1 << n_env) | flipped
                gate[idx, idx] = 0
                gate[new_idx, idx] = 1
                gate[idx, new_idx] = 1 if new_idx != idx else 0
                # Fix: clear the original mapping
                if new_idx != idx:
                    gate[new_idx, new_idx] = 0
        v_full = gate @ v_full
    
    # Partial trace over all environment qubits
    rho_full = np.outer(v_full, v_full.conj())
    rho_sys = partial_trace_B(rho_full, dA=2, dB=d_env)
    coh = abs(rho_sys[0, 1])
    pur = np.real(np.trace(rho_sys @ rho_sys))
    coherences.append(coh)
    purities.append(pur)
    print(f"  {n_env} env qubits: |ρ₀₁| = {coh:.6f}, purity = {pur:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(n_env_list, coherences, 'o-', color='crimson', lw=2)
axes[0].set_xlabel('Number of environment qubits')
axes[0].set_ylabel('|ρ₀₁| (coherence)')
axes[0].set_title('Decoherence: coherences die with environment size')
axes[0].grid(alpha=0.3)

axes[1].plot(n_env_list, purities, 'o-', color='steelblue', lw=2)
axes[1].axhline(abs(alpha)**4 + abs(beta)**4, color='green', ls='--', label='Fully decohered purity')
axes[1].set_xlabel('Number of environment qubits')
axes[1].set_ylabel('Purity Tr(ρ²)')
axes[1].set_title('Purity drops toward fully decohered value')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('The arrow of time: more environment → less reversibility', fontsize=13)
plt.tight_layout(); plt.show()